# Sales Trend Visualization

**CODTECH IT Solutions — Data Analytics Internship — Task 1**

Intern: Chakiri Subramanyam | Intern ID: CITS5137

## 1. Generate the Dataset

In [ ]:
"""
generate_data.py
Generates a realistic synthetic sales dataset for the Sales Trend
Visualization task (Codtech IT Solutions - Data Analytics Internship, Task 1).
"""

import numpy as np
import pandas as pd

np.random.seed(42)

# 2 years of daily data
date_range = pd.date_range(start="2024-01-01", end="2025-12-31", freq="D")

regions = ["North", "South", "East", "West"]
categories = ["Electronics", "Clothing", "Groceries", "Furniture", "Toys"]

rows = []
for date in date_range:
    # seasonal effect: more sales in Nov-Dec, dip in Feb
    month = date.month
    seasonal_factor = 1.0
    if month in [11, 12]:
        seasonal_factor = 1.6
    elif month == 2:
        seasonal_factor = 0.75

    # weekend boost
    weekend_factor = 1.3 if date.weekday() >= 5 else 1.0

    # slight upward yearly trend
    year_factor = 1.0 + (date.year - 2024) * 0.12

    for region in regions:
        for category in categories:
            base = np.random.normal(loc=500, scale=120)
            units = max(0, np.random.poisson(8))
            revenue = max(
                0,
                base * seasonal_factor * weekend_factor * year_factor
                * np.random.uniform(0.8, 1.2),
            )
            rows.append(
                {
                    "Date": date,
                    "Region": region,
                    "Category": category,
                    "Units_Sold": units,
                    "Revenue": round(revenue, 2),
                }
            )

df = pd.DataFrame(rows)
df.to_csv("data/sales_data.csv", index=False)
print(f"Generated {len(df):,} rows -> data/sales_data.csv")


## 2. Load Data, Analyze Trends, and Plot Graphs

In [ ]:
"""
sales_trend_visualization.py
Task 1: Sales Trend Visualization
Codtech IT Solutions - Data Analytics Internship

Loads the sales dataset, performs trend analysis, and generates a set of
visualizations (line, bar, heatmap, pie) saved to the output/ folder.
"""

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

DATA_PATH = "data/sales_data.csv"
OUT_DIR = "output"

# ---------------------------------------------------------------
# 1. Load & prepare data
# ---------------------------------------------------------------
df = pd.read_csv(DATA_PATH, parse_dates=["Date"])
df["Month"] = df["Date"].dt.to_period("M").dt.to_timestamp()
df["Year"] = df["Date"].dt.year
df["MonthName"] = df["Date"].dt.strftime("%b")

print("Dataset shape:", df.shape)
print(df.head())

# ---------------------------------------------------------------
# 2. Monthly revenue trend (overall)
# ---------------------------------------------------------------
monthly = df.groupby("Month")["Revenue"].sum().reset_index()

plt.figure(figsize=(11, 5))
plt.plot(monthly["Month"], monthly["Revenue"], marker="o", color="#1f3a5f", linewidth=2)
plt.title("Monthly Sales Revenue Trend (2024-2025)", fontsize=14, fontweight="bold")
plt.xlabel("Month")
plt.ylabel("Total Revenue ($)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/01_monthly_revenue_trend.png")
plt.close()

# ---------------------------------------------------------------
# 3. Revenue by category (bar chart)
# ---------------------------------------------------------------
cat_rev = df.groupby("Category")["Revenue"].sum().sort_values(ascending=False).reset_index()

plt.figure(figsize=(9, 5))
sns.barplot(data=cat_rev, x="Category", y="Revenue", hue="Category",
            palette="crest", legend=False)
plt.title("Total Revenue by Product Category", fontsize=14, fontweight="bold")
plt.xlabel("Category")
plt.ylabel("Total Revenue ($)")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/02_revenue_by_category.png")
plt.close()

# ---------------------------------------------------------------
# 4. Revenue by region (pie chart)
# ---------------------------------------------------------------
region_rev = df.groupby("Region")["Revenue"].sum()

plt.figure(figsize=(7, 7))
plt.pie(
    region_rev,
    labels=region_rev.index,
    autopct="%1.1f%%",
    startangle=90,
    colors=sns.color_palette("Set2"),
)
plt.title("Revenue Share by Region", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/03_revenue_share_by_region.png")
plt.close()

# ---------------------------------------------------------------
# 5. Month x Category heatmap (seasonality)
# ---------------------------------------------------------------
pivot = df.pivot_table(
    index="MonthName", columns="Category", values="Revenue", aggfunc="sum"
)
month_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
pivot = pivot.reindex(month_order)

plt.figure(figsize=(10, 6))
sns.heatmap(pivot, cmap="YlGnBu", annot=False, fmt=".0f", linewidths=0.3)
plt.title("Revenue Heatmap: Month vs Category", fontsize=14, fontweight="bold")
plt.xlabel("Category")
plt.ylabel("Month")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/04_month_category_heatmap.png")
plt.close()

# ---------------------------------------------------------------
# 6. Year-over-year comparison
# ---------------------------------------------------------------
yoy = df.groupby(["MonthName", "Year"])["Revenue"].sum().reset_index()
yoy_pivot = yoy.pivot(index="MonthName", columns="Year", values="Revenue").reindex(month_order)

plt.figure(figsize=(11, 5))
for year in yoy_pivot.columns:
    plt.plot(yoy_pivot.index, yoy_pivot[year], marker="o", label=str(year), linewidth=2)
plt.title("Year-over-Year Monthly Revenue Comparison", fontsize=14, fontweight="bold")
plt.xlabel("Month")
plt.ylabel("Total Revenue ($)")
plt.legend(title="Year")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/05_yoy_comparison.png")
plt.close()

# ---------------------------------------------------------------
# 7. Summary stats -> CSV
# ---------------------------------------------------------------
summary = pd.DataFrame({
    "Total_Revenue": [df["Revenue"].sum()],
    "Total_Units_Sold": [df["Units_Sold"].sum()],
    "Avg_Daily_Revenue": [monthly["Revenue"].mean()],
    "Best_Month": [monthly.loc[monthly["Revenue"].idxmax(), "Month"].strftime("%Y-%m")],
    "Top_Category": [cat_rev.iloc[0]["Category"]],
    "Top_Region": [region_rev.idxmax()],
})
summary.to_csv(f"{OUT_DIR}/summary_stats.csv", index=False)

print("\nAll charts saved to the 'output/' folder.")
print(summary.T)
